# AAPL headline relevance, tone, and forward-return check

This notebook classifies each AAPL headline once with a five-shot prompt as **relevant** or **not relevant** to the ticker's stock-price drivers, then examines the tone of the relevant headlines against later AAPL returns.

The model sees only the target ticker, publication date, and article title. It does **not** receive Alpha Vantage ticker relevance, sentiment, topics, article summaries, price data, returns, or events. The `tickers` and `topics` dataset partitions are never downloaded.

This is not model training or fine-tuning: `n-shot` means adding `n` completed examples to the prompt at inference time. Run this on a CUDA machine; it intentionally refuses CPU inference.

In [28]:
from __future__ import annotations

import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download

pd.set_option("display.max_colwidth", 160)
pd.set_option('display.max_rows', 1000)

DATASET_REPO = "cookekieran/alphavantage-market-news"
MONTHS = pd.period_range("2023-01", "2026-06", freq="M").astype(str).tolist()
TICKER = "AAPL"
SAMPLE_SIZE = None  # None means every AAPL article in the configured months.
SHOT_COUNTS = [5]  # The requested five-shot filter; every headline is parsed once.
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for candidate in (PROJECT_ROOT / ".env", PROJECT_ROOT / "env.txt"):
    if candidate.exists():
        load_dotenv(candidate)

HF_TOKEN = os.getenv("HF_TOKEN")
print({"dataset": DATASET_REPO, "month": MONTHS, "ticker": TICKER, "sample_size": SAMPLE_SIZE, "model": MODEL_ID})


python-dotenv could not parse statement starting at line 3


{'dataset': 'cookekieran/alphavantage-market-news', 'month': ['2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10'], 'ticker': 'AAPL', 'sample_size': 500, 'model': 'Qwen/Qwen2.5-7B-Instruct'}


## 1. Load only article titles and publication dates

`requested_entity` is used only to retrieve a manageable ticker-specific sample. It is not passed to the model. This is deliberately different from the original notebook's join to Alpha Vantage ticker relevance and sentiment.

In [29]:
article_frames = []

for month in MONTHS:
    articles_path = Path(
        hf_hub_download(
            repo_id=DATASET_REPO,
            repo_type="dataset",
            filename=f"articles/{month}.parquet",
            token=HF_TOKEN,
        )
    )

    month_articles = pd.read_parquet(articles_path)
    article_frames.append(month_articles)

articles = pd.concat(article_frames, ignore_index=True)

required_columns = {"article_uid", "requested_entity", "time_published", "title"}
missing_columns = required_columns - set(articles.columns)
assert not missing_columns, f"Article data is missing: {sorted(missing_columns)}"

article_sample = (
    articles.loc[
        articles["requested_entity"].eq(TICKER),
        ["article_uid", "time_published", "title"],
    ]
    .dropna(subset=["time_published", "title"])
    .drop_duplicates("article_uid")
    .sort_values(["time_published", "article_uid"])
    .reset_index(drop=True)
)
if SAMPLE_SIZE is not None:
    article_sample = article_sample.head(SAMPLE_SIZE).reset_index(drop=True)

article_sample["time_published"] = pd.to_datetime(
    article_sample["time_published"], utc=True
)

display(article_sample[["time_published", "title"]])

,time_published,title
0,2023-01-02 04:52:00+00:00,What is TSMC?
1,2023-01-03 04:06:10+00:00,"If You Invested $100 in Berkshire Hathaway in 1965, This Is How Much You Would Have Today"
2,2023-01-03 04:06:10+00:00,"If You Invested $100 in Berkshire Hathaway in 1965, This Is How Much You Would Have Today"
3,2023-01-03 12:05:00+00:00,Tuesday's ETF with Unusual Volume: IWL
4,2023-01-04 09:26:00+00:00,Salesforce to lay off 10% of workforce to cut costs amid economic downturn
5,2023-01-04 11:52:00+00:00,Apple to sign Luxshare for iPhone production in China - FT
6,2023-01-05 03:59:02+00:00,Apple brings AI narration to audiobooks
7,2023-01-05 15:29:00+00:00,"Android has a response to iPhone’s satellite connection, thanks to Qualcomm"
8,2023-01-06 04:25:43+00:00,Nordstrom appoints Atticus Tysen to its board of directors
9,2023-01-06 04:25:59+00:00,Qualcomm partners with Iridium to develop satellite connectivity


In [30]:
# articles_path = Path(hf_hub_download(
#     repo_id=DATASET_REPO,
#     repo_type="dataset",
#     filename=f"articles/{MONTHS}.parquet",
#     token=HF_TOKEN,
# ))
# articles = pd.read_parquet(articles_path)
# required_columns = {"article_uid", "requested_entity", "time_published", "title"}
# missing_columns = required_columns - set(articles.columns)
# assert not missing_columns, f"Article partition is missing: {sorted(missing_columns)}"

# article_sample = (
#     articles.loc[articles["requested_entity"].eq(TICKER), ["article_uid", "time_published", "title"]]
#     .dropna(subset=["time_published", "title"])
#     .drop_duplicates("article_uid")
#     .sort_values(["time_published", "article_uid"])
#     .head(SAMPLE_SIZE)
#     .reset_index(drop=True)
# )
# article_sample["time_published"] = pd.to_datetime(article_sample["time_published"], utc=True)
# assert len(article_sample), f"No articles found for {TICKER} in {MONTH}."
# with pd.option_context('display.max_rows', SAMPLE_SIZE):
#     display(article_sample[["time_published", "title"]])


## 2. Fixed relevance demonstrations

These are hand-written, task-level demonstrations rather than examples selected from the test sample. They are ordered once and nested: the 3-shot prompt contains examples 1–3, while the 5-shot prompt contains examples 1–5. This makes shot count the only intentional prompt change.

In [31]:
FEW_SHOT_EXAMPLES = [
    {
        "date": "2023-01-03",
        "title": "Apple faces EU antitrust investigation over App Store practices",
        "label": "relevant",
    },
    {
        "date": "2023-01-04",
        "title": "Technology ETF increases holdings in Apple, Microsoft and Nvidia",
        "label": "not relevant",
    },
    {
        "date": "2023-01-05",
        "title": "Apple supplier reports disruption to iPhone component production",
        "label": "relevant",
    },
    {
        "date": "2023-01-06",
        "title": "Five mega-cap technology stocks investors are watching this week",
        "label": "not relevant",
    },
    {
        "date": "2023-01-09",
        "title": "Apple cuts iPhone production forecast after weaker demand report",
        "label": "relevant",
    },
]

def format_example(example: dict) -> str:
    return (
        f"Publication date: {example['date']}\n"
        f"Headline: {example['title']}\n"
        f"Classification: {example['label']}"
    )

def build_relevance_prompt(ticker: str, published_at: pd.Timestamp, title: str, shots: int) -> str:
    assert 0 <= shots <= len(FEW_SHOT_EXAMPLES)
    examples = "\n\n".join(format_example(example) for example in FEW_SHOT_EXAMPLES[:shots])
    demonstrations = f"\n\nLABELLED EXAMPLES\n{examples}" if examples else ""
    return f"""You classify financial-news headlines for ticker {ticker}.

Classify a headline as relevant only when it describes a specific development
that could directly affect {ticker}'s revenue, demand, costs, supply chain,
products, strategy, financing, competitive position, or regulatory environment.
Classify it as not relevant when {ticker} is only in an ETF, index, stock list,
peer comparison, market roundup, investment recommendation, or generic sector
commentary without a direct company-specific mechanism.

Do not predict a return or use information outside the supplied date and headline.
Reply with exactly one label: relevant or not relevant.{demonstrations}

TARGET HEADLINE
Publication date: {published_at.date().isoformat()}
Headline: {title}
Classification:""".strip()

print(build_relevance_prompt(TICKER, article_sample.loc[0, "time_published"], article_sample.loc[0, "title"], shots=2))


You classify financial-news headlines for ticker AAPL.

Classify a headline as relevant only when it describes a specific development
that could directly affect AAPL's revenue, demand, costs, supply chain,
products, strategy, financing, competitive position, or regulatory environment.
Classify it as not relevant when AAPL is only in an ETF, index, stock list,
peer comparison, market roundup, investment recommendation, or generic sector
commentary without a direct company-specific mechanism.

Do not predict a return or use information outside the supplied date and headline.
Reply with exactly one label: relevant or not relevant.

LABELLED EXAMPLES
Publication date: 2023-01-03
Headline: Apple faces EU antitrust investigation over App Store practices
Classification: relevant

Publication date: 2023-01-04
Headline: Technology ETF increases holdings in Apple, Microsoft and Nvidia
Classification: not relevant

TARGET HEADLINE
Publication date: 2023-01-02
Headline: What is TSMC?
Classificatio

## 3. Superseded duplicate experiment

The active five-shot relevance run appears below. This placeholder prevents the old duplicate run from classifying the same headlines twice.

In [32]:
# Superseded duplicate experiment. The single five-shot relevance run is below.


Loading weights: 100%|██████████| 339/339 [00:04<00:00, 76.29it/s]


## 4. Superseded duplicate display

The active five-shot results are displayed after the single relevance run below.

In [33]:
# Superseded duplicate display. The active five-shot results are displayed after the single run below.


shots,article_uid,time_published,title,0_shot,1_shot,2_shot,3_shot,4_shot,5_shot
0,01c4874dbb77a5c035d5bcea5fdd3ecc,2023-07-25 09:00:00+00:00,Introducing Norton Genie - Real-Time AI-powered Scam Detection at Your Fingertips,not relevant,not relevant,not relevant,not relevant,not relevant,not relevant
1,02bd82b62c7bf1d151a0808e2b11fbe3,2023-01-10 04:03:16+00:00,Inari’s share price down over 4% as Apple plans to drop Broadcom chip from devices,relevant,not relevant,not relevant,not relevant,not relevant,not relevant
2,03a1f810e8d95ea922887e8504ff2adb,2023-04-06 01:40:33+00:00,General Motors hates your iPhone,not relevant,not relevant,not relevant,not relevant,not relevant,not relevant
3,0447c69db0ac6b359d47220f1c08bc4f,2023-09-12 20:24:00+00:00,"Apple unveils iPhone 15 Pro with titanium case, holds line on prices",relevant,relevant,relevant,relevant,relevant,relevant
4,05bda0abee450c9248ea6a0d7f7dbd0d,2023-07-14 16:32:00+00:00,Apple Inc. stock underperforms Friday when compared to competitors despite daily gains,not relevant,not relevant,not relevant,not relevant,not relevant,not relevant
5,08838071f9f5ad5de8e609dfdd8c006d,2023-09-21 14:35:48+00:00,Expedia Group builds generative AI into multiple products in its “Fall Release”,not relevant,not relevant,not relevant,not relevant,not relevant,not relevant
6,09c0d12926b75d72091155728511af52,2023-05-05 13:00:00+00:00,Warren Buffett's Berkshire Hathaway has been a fortress stock during recessions and bear markets. Here's how,not relevant,not relevant,not relevant,not relevant,not relevant,not relevant
7,0b93aaad46897662804d51438a1e9ae2,2023-02-17 20:30:12+00:00,Familiar Faces Powering Growth Stock Resurgence,not relevant,not relevant,not relevant,not relevant,not relevant,not relevant
8,0b94c9bf653f26e12a7496a3690e3102,2023-02-21 03:45:10+00:00,Major League Soccer and RBC Wealth Management Announce Multi-Year Partnership,not relevant,not relevant,not relevant,not relevant,not relevant,not relevant
9,0bc3b18faa9176599ab4853716f1a89e,2023-03-09 02:13:32+00:00,Apple and Foxconn efforts win labour reforms to advance Indian production plans - FT,relevant,relevant,relevant,relevant,relevant,relevant


Titles whose classification changes with shot count: 16


shots,time_published,title
1,2023-01-10 04:03:16+00:00,Inari’s share price down over 4% as Apple plans to drop Broadcom chip from devices
46,2023-07-13 23:07:08+00:00,Unlocking Business Potential with Apple’s Vision Pro
48,2023-04-27 16:20:00+00:00,Webull expands to Japan
59,2023-02-11 03:32:43+00:00,Apple hires its first Chief People Officer in executive reshuffle
62,2023-08-29 19:30:41+00:00,Apple partner Globalstar taps former Qualcomm executive Paul Jacobs as CEO
80,2023-08-11 22:45:59+00:00,TechInsights Teardown: Apple iPhone 14 Plus
85,2023-09-11 07:35:00+00:00,"Qualcomm will make Apple’s iPhone modems until 2026, and its stock surges"
101,2023-02-23 18:04:00+00:00,Apple shareholder moves to boot Al Gore from board of directors
106,2023-10-24 15:25:43+00:00,FDA Clears Omnipod 5 App for iPhones
146,2023-09-01 15:18:00+00:00,Globalstar taps SpaceX to launch Apple's Emergency SOS satellites


## How to judge whether few-shot prompting helped

First manually label this small fixed sample without looking at the model results. Then compare each shot-count column with those labels. A shift is not automatically an improvement: inspect whether the new label has a defensible direct ticker-specific mechanism in the headline. Keep the five demonstrations fixed while comparing shot counts; change and test them separately later if prompt wording or example selection becomes the next question.

## 3. Five-shot relevance filter: one pass over all AAPL headlines

In [34]:
FEW_SHOT_EXAMPLES = [
    {
        "date": "2023-01-03",
        "title": "Apple faces EU antitrust investigation over App Store practices",
        "label": "relevant",
    },
    {
        "date": "2023-01-04",
        "title": "Apple reports a sharp iPhone sales decline and cuts its revenue outlook",
        "label": "relevant",
    },
    {
        "date": "2023-01-05",
        "title": "Component supplier shares fall after reports that Apple may switch chip suppliers",
        "label": "not relevant",
    },
    {
        "date": "2023-01-06",
        "title": "Apple appoints a new Chief People Officer in an executive reshuffle",
        "label": "not relevant",
    },
    {
        "date": "2023-01-09",
        "title": "Apple TV+ renews a drama series for another season",
        "label": "not relevant",
    },
]


def format_example(example: dict) -> str:
    return (
        f"Publication date: {example['date']}\n"
        f"Headline: {example['title']}\n"
        f"Classification: {example['label']}"
    )

def build_relevance_prompt(ticker: str, published_at: pd.Timestamp, title: str, shots: int) -> str:
    assert 0 <= shots <= len(FEW_SHOT_EXAMPLES)
    examples = "\n\n".join(format_example(example) for example in FEW_SHOT_EXAMPLES[:shots])
    demonstrations = f"\n\nLABELLED EXAMPLES\n{examples}" if examples else ""
    return f"""You classify financial-news headlines for ticker {ticker}.

Classify a headline as relevant only when it describes a specific development
that could directly affect {ticker}'s revenue, demand, costs, supply chain,
products, strategy, financing, competitive position, or regulatory environment.
Classify it as not relevant when {ticker} is only in an ETF, index, stock list,
peer comparison, market roundup, investment recommendation, or generic sector
commentary without a direct company-specific mechanism.

Do not predict a return or use information outside the supplied date and headline.
Reply with exactly one label: relevant or not relevant.{demonstrations}

TARGET HEADLINE
Publication date: {published_at.date().isoformat()}
Headline: {title}
Classification:""".strip()

print(build_relevance_prompt(TICKER, article_sample.loc[0, "time_published"], article_sample.loc[0, "title"], shots=2))


You classify financial-news headlines for ticker AAPL.

Classify a headline as relevant only when it describes a specific development
that could directly affect AAPL's revenue, demand, costs, supply chain,
products, strategy, financing, competitive position, or regulatory environment.
Classify it as not relevant when AAPL is only in an ETF, index, stock list,
peer comparison, market roundup, investment recommendation, or generic sector
commentary without a direct company-specific mechanism.

Do not predict a return or use information outside the supplied date and headline.
Reply with exactly one label: relevant or not relevant.

LABELLED EXAMPLES
Publication date: 2023-01-03
Headline: Apple faces EU antitrust investigation over App Store practices
Classification: relevant

Publication date: 2023-01-04
Headline: Apple reports a sharp iPhone sales decline and cuts its revenue outlook
Classification: relevant

TARGET HEADLINE
Publication date: 2023-01-02
Headline: What is TSMC?
Classifica

In [35]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "CUDA GPU required. Do not run this notebook on CPU."
assert HF_TOKEN, "HF_TOKEN is required to download the configured DeepSeek model."

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype="auto",
    device_map="auto",
).eval()

def generate_label(prompt: str) -> tuple[str | None, str]:
    messages = [{"role": "user", "content": prompt}]
    model_input = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    encoded = tokenizer(model_input, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(generated[0, encoded["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    labels = re.findall(r"\bnot relevant\b|\brelevant\b", completion.lower())
    return (labels[-1] if labels else None), completion

records = []
for article in article_sample.itertuples(index=False):
    for shots in SHOT_COUNTS:
        prompt = build_relevance_prompt(TICKER, article.time_published, article.title, shots)
        label, raw_response = generate_label(prompt)
        records.append({
            "article_uid": article.article_uid,
            "time_published": article.time_published,
            "title": article.title,
            "shots": shots,
            "classification": label,
            "raw_response": raw_response,
        })

comparison_df = pd.DataFrame(records)
assert len(comparison_df) == len(article_sample) * len(SHOT_COUNTS)


Loading weights: 100%|██████████| 339/339 [00:03<00:00, 94.10it/s] 


In [36]:

classification_table = (
    comparison_df
    .pivot(
        index=["article_uid", "time_published", "title"],
        columns="shots",
        values="classification",
    )
    .rename(columns=lambda shots: f"{shots}_shot")
    .reset_index()
)
display(classification_table)

changed_titles = classification_table.loc[
    classification_table.filter(regex=r"_shot$").nunique(axis=1, dropna=False).gt(1),
    ["time_published", "title"],
]
print(f"Titles whose classification changes with shot count: {len(changed_titles)}")
display(changed_titles)


shots,article_uid,time_published,title,0_shot,1_shot,2_shot,3_shot,4_shot,5_shot
0,01c4874dbb77a5c035d5bcea5fdd3ecc,2023-07-25 09:00:00+00:00,Introducing Norton Genie - Real-Time AI-powered Scam Detection at Your Fingertips,not relevant,not relevant,not relevant,not relevant,not relevant,not relevant
1,02bd82b62c7bf1d151a0808e2b11fbe3,2023-01-10 04:03:16+00:00,Inari’s share price down over 4% as Apple plans to drop Broadcom chip from devices,relevant,not relevant,not relevant,not relevant,not relevant,not relevant
2,03a1f810e8d95ea922887e8504ff2adb,2023-04-06 01:40:33+00:00,General Motors hates your iPhone,not relevant,not relevant,not relevant,not relevant,not relevant,not relevant
3,0447c69db0ac6b359d47220f1c08bc4f,2023-09-12 20:24:00+00:00,"Apple unveils iPhone 15 Pro with titanium case, holds line on prices",relevant,relevant,relevant,relevant,relevant,relevant
4,05bda0abee450c9248ea6a0d7f7dbd0d,2023-07-14 16:32:00+00:00,Apple Inc. stock underperforms Friday when compared to competitors despite daily gains,not relevant,not relevant,not relevant,not relevant,not relevant,not relevant
5,08838071f9f5ad5de8e609dfdd8c006d,2023-09-21 14:35:48+00:00,Expedia Group builds generative AI into multiple products in its “Fall Release”,not relevant,not relevant,not relevant,not relevant,not relevant,not relevant
6,09c0d12926b75d72091155728511af52,2023-05-05 13:00:00+00:00,Warren Buffett's Berkshire Hathaway has been a fortress stock during recessions and bear markets. Here's how,not relevant,not relevant,not relevant,not relevant,not relevant,not relevant
7,0b93aaad46897662804d51438a1e9ae2,2023-02-17 20:30:12+00:00,Familiar Faces Powering Growth Stock Resurgence,not relevant,not relevant,not relevant,not relevant,not relevant,not relevant
8,0b94c9bf653f26e12a7496a3690e3102,2023-02-21 03:45:10+00:00,Major League Soccer and RBC Wealth Management Announce Multi-Year Partnership,not relevant,not relevant,not relevant,not relevant,not relevant,not relevant
9,0bc3b18faa9176599ab4853716f1a89e,2023-03-09 02:13:32+00:00,Apple and Foxconn efforts win labour reforms to advance Indian production plans - FT,relevant,relevant,relevant,relevant,relevant,relevant


Titles whose classification changes with shot count: 31


shots,time_published,title
1,2023-01-10 04:03:16+00:00,Inari’s share price down over 4% as Apple plans to drop Broadcom chip from devices
38,2023-01-05 03:59:02+00:00,Apple brings AI narration to audiobooks
44,2023-01-13 03:59:02+00:00,Apple CEO Tim Cook to take a more than 40 percent pay cut
46,2023-07-13 23:07:08+00:00,Unlocking Business Potential with Apple’s Vision Pro
48,2023-04-27 16:20:00+00:00,Webull expands to Japan
59,2023-02-11 03:32:43+00:00,Apple hires its first Chief People Officer in executive reshuffle
62,2023-08-29 19:30:41+00:00,Apple partner Globalstar taps former Qualcomm executive Paul Jacobs as CEO
80,2023-08-11 22:45:59+00:00,TechInsights Teardown: Apple iPhone 14 Plus
85,2023-09-11 07:35:00+00:00,"Qualcomm will make Apple’s iPhone modems until 2026, and its stock surges"
93,2023-01-11 04:57:25+00:00,Apple debuts free tool to help businesses get noticed


# 5-shot filtered news tone versus subsequent AAPL returns

This is a deliberately simple, forward-looking check. It keeps **only** headlines marked `relevant` by the 5-shot run, labels their likely company-news tone as positive, negative, or neutral once, then reuses that fixed news table for every price horizon. It tests whether majority positive/negative tone in non-overlapping 3- or 6-month news windows agrees with AAPL's forward return after 3, 5, 15, 20, 35, 45, or 90 trading days.

The return starts at the first trading-session close after the news window and ends exactly the stated number of trading days later: `(end close / start close - 1) × 100`. `actual_direction` is `up` for a positive return, `down` for a negative return, and `flat` for zero. This is an association check, not evidence that headlines cause or reliably predict returns.

In [ ]:
# Use only the existing five-shot relevance decisions; no vendor sentiment is used.
five_shot_relevant = (
    comparison_df.loc[
        comparison_df["shots"].eq(5) & comparison_df["classification"].eq("relevant"),
        ["article_uid", "time_published", "title"],
    ]
    .drop_duplicates("article_uid")
    .sort_values("time_published")
    .reset_index(drop=True)
)

def build_tone_prompt(ticker: str, published_at: pd.Timestamp, title: str) -> str:
    return f"""Classify the likely company-news tone of this relevant headline for {ticker}.

Use positive for a development likely favourable to the company, negative for a development likely unfavourable to the company, and neutral when the direction is unclear or balanced. Do not use later price movement or outside information.

Publication date: {published_at.date().isoformat()}
Headline: {title}
Tone (reply with exactly one label: positive, negative, or neutral):"""

def generate_tone(prompt: str) -> tuple[str | None, str]:
    messages = [{"role": "user", "content": prompt}]
    model_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    encoded = tokenizer(model_input, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **encoded, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    completion = tokenizer.decode(
        generated[0, encoded["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    labels = re.findall(r"\bpositive\b|\bnegative\b|\bneutral\b", completion.lower())
    return (labels[-1] if labels else None), completion

tone_records = []
for article in five_shot_relevant.itertuples(index=False):
    tone, raw_tone_response = generate_tone(
        build_tone_prompt(TICKER, article.time_published, article.title)
    )
    tone_records.append({
        "article_uid": article.article_uid,
        "time_published": article.time_published,
        "title": article.title,
        "tone": tone,
        "raw_tone_response": raw_tone_response,
    })

five_shot_tone = pd.DataFrame(tone_records)
print({"five_shot_relevant_headlines": len(five_shot_tone), "tone_counts": five_shot_tone["tone"].value_counts(dropna=False).to_dict()})
display(five_shot_tone[["time_published", "title", "tone"]])


In [ ]:
import matplotlib.pyplot as plt

PRICE_REPO = "cookekieran/mag7_prices"
PRICE_FILENAME = "mag7_daily_prices.parquet"

price_path = Path(hf_hub_download(
    repo_id=PRICE_REPO, repo_type="dataset", filename=PRICE_FILENAME, token=HF_TOKEN
))
prices_raw = pd.read_parquet(price_path)

def find_column(frame: pd.DataFrame, candidates: set[str]) -> str:
    lookup = {str(column).strip().lower().replace(" ", "_"): column for column in frame.columns}
    match = next((lookup[name] for name in candidates if name in lookup), None)
    assert match is not None, f"Expected one of {sorted(candidates)}; found {list(frame.columns)}"
    return match

price_date_column = find_column(prices_raw, {"date", "timestamp", "datetime", "time"})
price_ticker_column = find_column(prices_raw, {"ticker", "symbol"})
price_close_column = find_column(prices_raw, {"adjusted_close", "adj_close", "close"})

aapl_prices = (
    prices_raw.loc[prices_raw[price_ticker_column].astype(str).str.upper().eq(TICKER)]
    .assign(date=lambda frame: pd.to_datetime(frame[price_date_column], utc=True).dt.tz_localize(None))
    .rename(columns={price_close_column: "close"})[["date", "close"]]
    .dropna()
    .drop_duplicates("date")
    .sort_values("date")
    .query("date >= '2023-01-01' and date <= '2026-12-31'")
    .reset_index(drop=True)
)
assert not aapl_prices.empty, "No AAPL prices were found in the configured price file."

monthly_price = aapl_prices.set_index("date")["close"].resample("ME").last()
fig, ax = plt.subplots(figsize=(12, 4))
monthly_price.plot(ax=ax, color="#1f77b4", linewidth=2)
ax.set(title="AAPL month-end close (2023--2026 available data)", xlabel="", ylabel="Close price")
ax.grid(axis="y", alpha=0.3)
plt.show()
print({"price_coverage": (str(aapl_prices["date"].min().date()), str(aapl_prices["date"].max().date())), "trading_days": len(aapl_prices)})


In [ ]:
# News is classified above exactly once. This cell only reuses five_shot_tone and price data.
MIN_DIRECTIONAL_ARTICLES = 5  # avoids calling a one-headline window "mainly" positive or negative
FORWARD_TRADING_DAYS = [3, 5, 15, 20, 35, 45, 90]
NEWS_WINDOW_MONTHS = [3, 6]
ANALYSIS_START = pd.Timestamp("2023-01-01")
ANALYSIS_END = pd.Timestamp("2026-06-01")

tone_for_windows = five_shot_tone.dropna(subset=["tone"]).copy()
tone_for_windows["month"] = tone_for_windows["time_published"].dt.tz_localize(None).dt.to_period("M").dt.to_timestamp()
price_dates = aapl_prices["date"].reset_index(drop=True)
price_closes = aapl_prices["close"].reset_index(drop=True)

def non_overlapping_windows(months: int):
    starts = pd.date_range(ANALYSIS_START, ANALYSIS_END, freq=pd.DateOffset(months=months))
    for start in starts:
        end = start + pd.DateOffset(months=months) - pd.Timedelta(days=1)
        if end <= ANALYSIS_END:
            yield start, end

def first_price_index_after(day: pd.Timestamp) -> int | None:
    positions = price_dates.index[price_dates.gt(day)]
    return None if len(positions) == 0 else int(positions[0])

def evaluate_window(months: int, window_start: pd.Timestamp, window_end: pd.Timestamp) -> list[dict]:
    in_window = tone_for_windows.loc[
        tone_for_windows["month"].between(window_start, window_end.to_period("M").to_timestamp())
    ]
    counts = in_window["tone"].value_counts()
    positive = int(counts.get("positive", 0))
    negative = int(counts.get("negative", 0))
    directional = positive + negative
    signal = (
        "bullish" if directional >= MIN_DIRECTIONAL_ARTICLES and positive > negative
        else "bearish" if directional >= MIN_DIRECTIONAL_ARTICLES and negative > positive
        else "mixed / insufficient"
    )

    # First close after the news window; h trading days ahead means close[t+h] / close[t] - 1.
    start_index = first_price_index_after(window_end)
    if start_index is None:
        return []

    rows = []
    for horizon_days in FORWARD_TRADING_DAYS:
        end_index = start_index + horizon_days
        if end_index >= len(price_dates):
            continue  # The future price period is not available yet.
        forward_return = price_closes.iloc[end_index] / price_closes.iloc[start_index] - 1
        actual_direction = "up" if forward_return > 0 else "down" if forward_return < 0 else "flat"
        rows.append({
            "news_window_months": months,
            "news_window": f"{window_start:%Y-%m-%d} to {window_end:%Y-%m-%d}",
            "positive": positive,
            "negative": negative,
            "neutral": int(counts.get("neutral", 0)),
            "news_signal": signal,
            "forward_trading_days": horizon_days,
            "return_period": f"{price_dates.iloc[start_index]:%Y-%m-%d} to {price_dates.iloc[end_index]:%Y-%m-%d}",
            "forward_return_pct": round(forward_return * 100, 2),
            "actual_direction": actual_direction,
            "direction_agrees": (signal == "bullish" and actual_direction == "up") or (signal == "bearish" and actual_direction == "down"),
        })
    return rows

trend_rows = []
for months in NEWS_WINDOW_MONTHS:
    for window_start, window_end in non_overlapping_windows(months):
        trend_rows.extend(evaluate_window(months, window_start, window_end))

trend_check = pd.DataFrame(trend_rows)
testable = trend_check.loc[trend_check["news_signal"].ne("mixed / insufficient")].copy()
agreement_summary = (
    testable.groupby(["news_window_months", "forward_trading_days"], as_index=False)
    .agg(windows_tested=("direction_agrees", "size"), agreement_rate_pct=("direction_agrees", "mean"))
    .assign(agreement_rate_pct=lambda frame: (frame["agreement_rate_pct"] * 100).round(1))
    .sort_values(["news_window_months", "forward_trading_days"])
)

display(trend_check)
display(agreement_summary)
print("Agreement rate is the percentage of testable news windows where bullish news preceded an up return or bearish news preceded a down return.")
print("The price-horizon loop does not call the LLM or re-parse articles.")
